<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [29]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [30]:
# from qiskit_aer import AerSimulator
# simulator = AerSimulator()

# print(simulator.available_devices())

<h3>Download MQT Benchmark circuits.</h3>

We'll run this experiment on 11 different quantum circuits, including DJ, GHZ, GraphState, Grover No-Ancilla, QAOA, QFT, QGAN, QPE Exact, QPE Inexact, RealAmpRandom, and VQE.



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later. Missing code in SBFL is still a more recent topic even in classical programs.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate at any location with 0-10% survival scores

<h2 style="text-align: center;">CONTROL PANEL</h2>

In [31]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ALG_NAME = "graphstate"
ORIGINAL_QASM = f"{ALG_NAME}_5_qubits.qasm"                                                 #| The original algorithm QASM circuit file name.                           
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                            #| The folder containing the original QASM circuit file.                    
QASM_MUTANT_FOLDER = f"SBFL_circuits/QMutBenchMutants/Mutants_{ALG_NAME}_5_qubits/"         #| The folder containing all mutants related to the original QASM circuit.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                                   #| Number of times SB-QOPS will run to find a failing or passing test case.            
SHOTS = 20000                                                                               #| Number of shots for SB-QOPS to use to create the compact program specification.     
EPOCH = 80                                                                                  #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       
SBQOPS_TOL = 0.1                                                                            #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
LAMBDA_WEIGHT = 1                                                                           #| The amount to adjust the "weight difference" in the heuristic           #
LAMBDA_COMP = 1                                                                             #| The amount to weigh the "composition difference" in the heuristic       #
LAMBDA_SUPPORT = 1                                                                          #| The amount to weigh the "support overlap" in the heuristic              #
LAMBDA_DIAMETER = 1                                                                         #| The amount to weigh the "diameter difference" in the heuristic          #
LAMBDA_PHASE = 1                                                                            #| The amount to weigh the "phase" in the heuristic                        #
CUSTOM_WEIGHT = 0.25                                                                        #| The amount of weight the "dstar boost" has in the custom heuristic (WIP)#
ATOL = 1e-4                                                                                 #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   
MAX_TERMS = 100                                                                             #| The maximum number of terms to keep during Pauli propagation. If None, use all.     
SEARCH_STEP = 3                                                                             #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     
VERBOSE = 1                                                                                 #| A boolean value to determine if the program should print out detailed information.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all gates found in a given circuit

In [32]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Load correct circuit</h3>
Here we want to load the correct version of the circuit from MQT. We use this both to generate a PS_compact from/for SB-QOPS and to check results from SBFL.

In [33]:
correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
correct_list = construct_list(correct_list, correct_circuit, inverse=False)

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

Expect this cell to take up to a few hours, BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [34]:
# import QOPS as qops
# from QOPS_test import get_compact_program_specification_Z
# from pathlib import Path

# ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
# program_history = {}

# #program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases.
# # In a public-use environment, this would be provided by the user.
# program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

# #mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
# mutants = []
# mutants_names = []
# root = Path(QASM_MUTANT_FOLDER)
# for qasm_mutant in root.rglob("*.qasm"):
#     mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
#     mutants_names.append(qasm_mutant.name)

# for mutant_index,mutant in enumerate(mutants):
#     tester = qops.Circuit_Tester(CUT=mutant)
#     tester.set_applicable_families_Z(program_specification)
#     mutants_per_run = []
#     fitness_per_run = []
#     failing_testcases_per_run = []
#     history_per_run = []

#     print("NOW RUNNING TO FIND FAILING TESTS")
#     #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
#     for runs in range(RUNS):
#         killed = 0
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
#             if best_function > SBQOPS_TOL:
#                 killed = 1
#                 pauli = testcase
#                 fitness = best_function
#                 break
#         mutants_per_run.append(killed)
#         failing_testcases_per_run.append(pauli)
#         fitness_per_run.append(fitness)
#         history_per_run.append(history)

#     avg_score = np.mean(mutants_per_run)
#     avg_fitness = np.mean(fitness_per_run)

#     #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
#     passing_testcases_per_run = []

#     print("NOW RUNNING TO FIND PASSING TESTS")
#     for runs in range(RUNS):
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
#             if best_function < SBQOPS_TOL:
#                 pauli = testcase
#                 break
#         passing_testcases_per_run.append(pauli)

#     #Replace static program file with "program_name" when ready to test fully
#     """
#     ga_result: A pandas dataframe with the following columns
#         Program: Name of the mutant quantum circuit file
#         Path_To_Mutant: Path to the mutant file
#         Catch_Avg: The average number of times the mutant was identified by SB-QOPS
#         Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
#         Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
#         Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
#     """
#     ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
#     ga_result.to_csv(f'realamprandom_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis can vary widely in runtime. Some of the simplest circuits take seconds to analyze and only took upwards of 3-4 minutes to get results on the whole dataset (GHZ was the fastest taking 3 minutes to run). Other, more complex algorithms can take upwards of 6-10 minutes PER MUTANT (RealAmpRandom took longest so far with 14 hours).

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [35]:
from heisenbergTree import *

LAMBDAS = [LAMBDA_WEIGHT, LAMBDA_COMP, LAMBDA_SUPPORT, LAMBDA_DIAMETER, LAMBDA_PHASE]
ga_result = pd.read_csv(f'{ALG_NAME}_ga_result.csv')
custom_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
dstar_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
mutant_num = 0

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    raw_failing_testcases = format_test_case(ga_result, mutant, mutant_num, "failing_testcases")
    raw_passing_testcases = format_test_case(ga_result, mutant, mutant_num, "passing_testcases")

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    operation_list = construct_list(operation_list, circuit_inverse, True)

    forward_list = LinkedList()
    forward_list = construct_list(forward_list, forward_mutant, False)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_failing_testcases, 
                                          "fail", 
                                          LAMBDAS,
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)
    
    global_frame_pass = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_passing_testcases, 
                                          "pass", 
                                          LAMBDAS,
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)
    if VERBOSE:
        display(global_frame)

    custom_scores = custom_sbfl(global_frame, CUSTOM_WEIGHT)
    barinel_scores = barinel(global_frame)
    dstar_scores = dstar(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("custom SCORES")
        display(custom_scores)
        print("DSTAR SCORES")
        display(dstar_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_mutant, correct_circuit)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    #Calculate best, worst, and average EXAM for Barinel, custom, and DStar
    barinel_best_exam, barinel_worst_exam, barinel_average_exam = calc_exams(barinel_scores, error_gate_location, circuit_inverse)
    custom_best_exam, custom_worst_exam, custom_average_exam = calc_exams(custom_scores, error_gate_location, circuit_inverse)
    dstar_best_exam, dstar_worst_exam, dstar_average_exam = calc_exams(dstar_scores, error_gate_location, circuit_inverse)

    # Here we store the results from the HBFL analysis for both barinel and custom into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_best_exam, barinel_worst_exam, barinel_average_exam]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.to_csv(f'{ALG_NAME}_barinel_sbfl_result.csv', index=False)

    custom_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,custom_best_exam, custom_worst_exam, custom_average_exam]],columns=custom_sbfl_result.columns),custom_sbfl_result],ignore_index=True)
    custom_sbfl_result.to_csv(f'{ALG_NAME}_custom_sbfl_result.csv', index=False)

    dstar_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,dstar_best_exam, dstar_worst_exam, dstar_average_exam]],columns=dstar_sbfl_result.columns),dstar_sbfl_result],ignore_index=True)
    dstar_sbfl_result.to_csv(f'{ALG_NAME}_dstar_sbfl_result.csv', index=False)

    mutant_num += 1

if VERBOSE:
    display(barinel_sbfl_result)
    display(custom_sbfl_result)
    

Now evolving the following mutant:  AddGate_ccx_inGap_11_.qasm


100%|██████████| 10/10 [00:03<00:00,  3.21it/s]


,ccx 10,cz 9,cz 8,h 7,cz 6,cz 5,h 4,h 3,cz 2,h 1,h 0,pass/fail
0,0.450481,0.0,0.068274,0.023995,0.0,0.0,0.011479,0.012420,0.016393,0.011114,0.014266,fail
1,0.446898,0.0,0.057740,0.024625,0.0,0.0,0.009564,0.011517,0.015297,0.011712,0.013451,fail
2,0.435293,0.0,0.051863,0.027951,0.0,0.0,0.012524,0.013176,0.013969,0.010828,0.015629,fail
3,0.499356,0.0,0.078664,0.025135,0.0,0.0,0.012075,0.014451,0.022811,0.013372,0.014534,pass
4,0.529376,0.0,0.067147,0.027263,0.0,0.0,0.012448,0.014991,0.020321,0.012667,0.015407,pass
5,0.482722,0.0,0.076173,0.030895,0.0,0.0,0.014390,0.014859,0.018578,0.012429,0.016934,pass
6,0.473074,0.0,0.057610,0.023970,0.0,0.0,0.008391,0.012285,0.017400,0.009438,0.011889,pass
7,0.364630,0.0,0.060993,0.018018,0.0,0.0,0.007916,0.009480,0.016152,0.008322,0.009130,pass
8,0.393551,0.0,0.064003,0.020275,0.0,0.0,0.011886,0.010927,0.014914,0.008342,0.010945,pass
9,0.517406,0.0,0.064628,0.034064,0.0,0.0,0.013533,0.015986,0.020841,0.016173,0.017988,pass


[[1.01408549        nan 1.15147692 1.09510398        nan        nan
  1.11934684 1.06508692 1.07709627 1.04403488 1.08172182]]
BARINEL SCORES


d:\GitHub\heisenberg-SBFL-experiment\heisenbergUtilities.py:326: RuntimeWarning: invalid value encountered in divide
  dstar_boost = max_fail.values/avg_fail.values


...

DSTAR SCORES


,ccx 3,h 8,h 4,h 5,h 0,h 1,cz 10,cz 9,cz 7,cz 6,cz 2
sum,0.007558,0.00186,0.000857,0.000488,0.000308,0.000186,0.0,0.0,0.0,0.0,0.0


ERROR GATE LOCATION:
3


,Program,path_to_mutant,best_exam_score,worst_exam_score,avg_exam_score
0,AddGate_ccx_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,0.545455,0.545455
1,AddGate_id_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
2,AddGate_p_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
3,AddGate_ccx_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.272727,0.272727,0.272727
4,AddGate_ry_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.272727,0.363636,0.318182
...,...,...,...,...,...
187,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.272727,0.363636,0.318182
188,AddGate_ry_inGap_15_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.181818,0.181818,0.181818
189,AddGate_sx_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
190,AddGate_p_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727


,Program,path_to_mutant,best_exam_score,worst_exam_score,avg_exam_score
0,AddGate_ccx_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,0.545455,0.545455
1,AddGate_id_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
2,AddGate_p_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
3,AddGate_ccx_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.363636,0.363636,0.363636
4,AddGate_ry_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.181818,0.272727,0.227273
...,...,...,...,...,...
187,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.363636,0.363636,0.363636
188,AddGate_ry_inGap_15_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.272727,0.272727,0.272727
189,AddGate_sx_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
190,AddGate_p_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_graphst...,0.545455,1.0,0.772727
